# 🚀 Direct QA Evaluation on Model-Generated Spatial Scene Graphs
**Model**: `Qwen/Qwen3-VL-2B-Instruct`  
**Scene Graphs**: 100 Full-Image Scene Graphs Generated with **Few-Shot + Embedded Spatial CoT**  
**Strategy**: Direct Prompting (No CoT, max_new_tokens=32)  
**Hardware**: Google Colab Free Tier (Tesla T4 GPU, ~20s total inference)

In [ ]:
# 1. Install dependencies
!pip install -q qwen-vl-utils "transformers>=4.49.0" accelerate pillow pandas

In [ ]:
# 2. Check GPU & Load Qwen3-VL-2B-Instruct
import os, sys, time, json, re, gzip, base64, torch
from transformers import AutoProcessor, AutoModelForImageTextToText

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Hardware: {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'} | Device: {device}")
if device != "cuda":
    print("[WARN] Please enable GPU: Runtime -> Change runtime type -> T4 GPU")

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
print(f"[INFO] Loading {MODEL_ID} in float16...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
print("[INFO] Model loaded successfully!")

In [ ]:
# 3. Load Embedded Benchmark Data (100 Items with Spatial Scene Graphs)
SAMPLES_B64 = "..."
SAMPLES = json.loads(gzip.decompress(base64.b64decode(SAMPLES_B64.encode('ascii'))).decode('utf-8'))
print(f"[INFO] Successfully loaded {len(SAMPLES)} benchmark items.")

# Print sample preview
sample_preview = SAMPLES[1]  # Image 002 (Airplane)
print(f"Sample Preview (Image {sample_preview['image_number']}):")
print(f"  Question: {sample_preview['question']}")
print(f"  Ground Truth: {sample_preview['ground_truth']}")
print(f"  Objects ({len(sample_preview['objects'])}): {[o['label'] for o in sample_preview['objects'][:4]]}...")
print(f"  Relations ({len(sample_preview['relations'])}): {[f'{r["subject"]} {r["relation"]} {r["object"]}' for r in sample_preview['relations'][:3]]}...")

In [ ]:
# 4. Prompt Construction & Output Extraction
def format_sg(item):
    objects = item.get("objects", [])
    relations = item.get("relations", [])
    obj_lines = []
    for obj in objects:
        lbl = obj.get("label", obj.get("name", "unknown"))
        attrs = obj.get("attributes", [])
        attr_str = f"attributes: [{', '.join(attrs)}]" if attrs else "no attributes"
        obj_lines.append(f"  - {lbl} ({attr_str})")

    rel_lines = []
    for rel in relations:
        s = rel.get("subject", "")
        r = rel.get("relation", "")
        o = rel.get("object", "")
        rel_lines.append(f'  - "{s}" {r} "{o}"')
        
    return len(objects), "\n".join(obj_lines) if obj_lines else "  None", len(relations), "\n".join(rel_lines) if rel_lines else "  None"

def build_direct_prompt(item):
    num_obj, obj_str, num_rel, rel_str = format_sg(item)
    q = item.get("question", "")
    return (
        "You are given a scene graph representation of an image containing objects and their relationships, followed by a question.\n\n"
        f"Scene Graph:\nObjects ({num_obj}):\n{obj_str}\n\n"
        f"Relationships ({num_rel}):\n{rel_str}\n\n"
        f"Question: {q}\n\n"
        "Instructions: Based strictly on the scene graph provided above, answer the question accurately. "
        "Provide only the direct concise answer (a few words or a single term) without explanation or reasoning.\n"
        "Answer:"
    )

def extract_direct_answer(pred_text):
    clean = pred_text.strip()
    clean = re.sub(r"^(?:answer|the answer is|final answer):\s*", "", clean, flags=re.IGNORECASE).strip()
    first_line = clean.split("\n")[0].strip()
    first_line = re.sub(r"^(?:answer|the answer is|final answer):\s*", "", first_line, flags=re.IGNORECASE).strip()
    return first_line.rstrip(".")

In [ ]:
# 5. Run Evaluation Loop across 100 Items (~15-20 seconds on T4 GPU)
results = []
correct_count = 0
exact_count = 0
cat_stats = {
    "attr_relation": {"total": 0, "correct": 0, "exact": 0},
    "two_hop": {"total": 0, "correct": 0, "exact": 0},
    "spatial_conjunction": {"total": 0, "correct": 0, "exact": 0}
}

print("="*75)
print(f"RUNNING DIRECT EVALUATION ON SPATIAL SCENE GRAPHS (ALL {len(SAMPLES)} SAMPLES)")
print("="*75 + "\n")

t_start = time.time()

for i, item in enumerate(SAMPLES, 1):
    gt = str(item.get("ground_truth", "")).strip().lower()
    q = item["question"]
    t = item["template"]
    cat_stats[t]["total"] += 1
    
    prompt = build_direct_prompt(item)
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    
    chat_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[chat_text], return_tensors="pt").to(device)
    
    start_time = time.time()
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            repetition_penalty=1.15
        )
    elapsed = time.time() - start_time
    
    gen_ids = out_ids[0][len(inputs["input_ids"][0]):]
    raw_pred = processor.decode(gen_ids, skip_special_tokens=True).strip()
    parsed = extract_direct_answer(raw_pred)
    clean_pred = parsed.lower().rstrip(".").strip()
    
    is_exact = (clean_pred == gt)
    is_correct = is_exact or (gt in clean_pred) or (clean_pred in gt and len(clean_pred) > 1)
    
    if is_correct:
        correct_count += 1
        cat_stats[t]["correct"] += 1
    if is_exact:
        exact_count += 1
        cat_stats[t]["exact"] += 1
        
    status = "EXACT" if is_exact else ("CORRECT" if is_correct else "WRONG")
    
    results.append({
        "sample_index": item["sample_index"],
        "image_number": item["image_number"],
        "template": t,
        "question": q,
        "ground_truth": gt,
        "latency_sec": elapsed,
        "raw_output": raw_pred,
        "predicted_answer": parsed,
        "is_correct": is_correct,
        "is_exact": is_exact
    })
    
    if i % 10 == 0 or i == len(SAMPLES) or not is_correct:
        print(f"[{i:03d}/{len(SAMPLES)}] Img {item['image_number']:03d} | {t[:12]:12} | {elapsed:.2f}s | {status:7} | Pred: '{parsed}' | GT: '{gt}'")

total_time = time.time() - t_start
acc = (correct_count / len(SAMPLES)) * 100
exact_acc = (exact_count / len(SAMPLES)) * 100

print("\n" + "="*75)
print("BENCHMARK RESULTS (SPATIAL MODEL-GENERATED SCENE GRAPHS)")
print("="*75)
print(f"Overall Accuracy: {acc:.2f}% ({correct_count}/{len(SAMPLES)})")
print(f"Exact Match Accuracy: {exact_acc:.2f}% ({exact_count}/{len(SAMPLES)})")
print(f"Total Inference Time: {total_time:.2f}s (Avg: {total_time/len(SAMPLES):.3f}s/query)")
print("\nCategory Breakdown:")
for cat, st in cat_stats.items():
    tot = st["total"]
    c_acc = (st["correct"] / tot * 100) if tot > 0 else 0
    e_acc = (st["exact"] / tot * 100) if tot > 0 else 0
    print(f"  {cat:20}: {c_acc:6.2f}% Acc ({st['correct']}/{tot}) | Exact: {e_acc:6.2f}% ({st['exact']}/{tot})")
print("="*75)

# Save payload
payload = {
    "strategy": "Direct Prompting (Spatial Model-Generated Scene Graphs)",
    "total": len(SAMPLES),
    "correct": correct_count,
    "exact": exact_count,
    "accuracy": acc,
    "exact_accuracy": exact_acc,
    "total_time_seconds": total_time,
    "category_breakdown": cat_stats,
    "results": results
}
with open("spatial_sg_benchmark_results.json", "w") as f:
    json.dump(payload, f, indent=2)

# Copy to Google Drive if mounted
for drive_dir in ["/content/drive/MyDrive/RM", "/content/drive/My Drive/RM"]:
    if os.path.exists(drive_dir):
        shutil.copy2("spatial_sg_benchmark_results.json", os.path.join(drive_dir, "spatial_sg_benchmark_results.json"))
        print(f"[INFO] Copied results to {drive_dir}")

In [ ]:
# 6. Comparison Table with Previous Benchmarks
import pandas as pd

comparison = {
    "Strategy": [
        "Ground Truth SGs (Direct)",
        "Zero-Shot CoT (Images directly)",
        "Old Model-Generated SGs (Direct)",
        "NEW Spatial Model-Generated SGs (Direct)"
    ],
    "Overall Acc (%)": [79.0, 51.0, 25.0, acc],
    "Exact Match (%)": [77.0, 30.0, 17.0, exact_acc],
    "attr_relation (%)": [85.3, 67.6, 32.4, (cat_stats['attr_relation']['correct']/cat_stats['attr_relation']['total']*100)],
    "spatial_conjunction (%)": [84.8, 45.5, 27.3, (cat_stats['spatial_conjunction']['correct']/cat_stats['spatial_conjunction']['total']*100)],
    "two_hop (%)": [66.7, 39.4, 15.2, (cat_stats['two_hop']['correct']/cat_stats['two_hop']['total']*100)],
    "Avg Latency (s)": [0.17, 40.2, 0.17, (total_time/len(SAMPLES))]
}
df = pd.DataFrame(comparison)
display(df)